# Full testbed

In [4]:
import dataclasses

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from anb.agents import ANBAgent
from anb.metrics import RESPONSE_COLUMNS
from anb.model import ModelSpec
from anb.networks import ErdosRenyiNetwork, Network
from anb.testbed import PressureSchedule, Testbed, TestbedConfig

## Set config

In [ ]:
N_AGENTS = 100
M = 10
TAU = 1
LINK_PROB = 10 / N_AGENTS
BETA = 3.0
RHO = 1/3

# The sweep from ANBmodel.py's __main__ block: (init_w, eps, mu, fixedBNat100)
SETTINGS = [
    (0.2, 0.0, 0.0, False),  # static, low edge weights
    (0.8, 0.0, 0.0, False),  # static, high edge weights
    (0.2, 1.0, 0.0, True),   # adaptive until external pressure
    (0.2, 1.0, 0.0, False),  # fully adaptive
]
PRESSURES = [0, 1, 2, 4, 8, 16]
SEEDS = range(100)


In [6]:
def make_spec(init_w, eps, mu, fixedBNat100):
    # fixedBNat100 is learning_until = 101 in the package (see docu/validation.md, deviation 2)
    agent = ANBAgent(beta=BETA, eps=eps, mu=mu, init_w=init_w, tau=TAU, n_beliefs=M,
                     learning_until=101 if fixedBNat100 else None)
    return ModelSpec(agent=agent, network_cls=ErdosRenyiNetwork,
                     network_kwargs={"link_prob": LINK_PROB}, n_agents=N_AGENTS, rho=RHO)


# One ensemble per (setting, pressure), like legacy's param_combis × seeds. n_jobs=-2 = all cores but one.
results = {}
for setting in SETTINGS:
    spec = make_spec(*setting)
    for pressure in PRESSURES:
        config = TestbedConfig(PressureSchedule.single_event(strength=pressure))
        results[(*setting, pressure)] = Testbed(spec, config).run_ensemble(SEEDS, n_jobs=-2)

# Legacy-named CSVs in ../simOut, where the analysis scripts look (ANBmodel.py writes to ../simOut_old_model):
for runs in results.values():
    for r in runs:
        r.save("../simOut")
